In [ ]:
FIGURE = 'Fig.2'
from pathlib import Path
from contextlib import contextmanager
import os
import sys
import gc
import mmap
import json
import hashlib
import shutil
import numpy as np
import pandas as pd
import netCDF4 as nc
_NETCDF_DATASET = nc.Dataset
HERE = Path.cwd().resolve()
if not (HERE / 'Inputs').is_dir():
    for candidate in [HERE / 'Figure' / FIGURE, HERE / FIGURE, HERE / 'Data Processing' if FIGURE in ['SSMIrz','Rmax_Ts'] else HERE]:
        if (candidate / 'Inputs').is_dir():
            HERE = candidate
            break
INPUT_DIR = HERE / 'Inputs'
OUTPUT_DIR = HERE / 'Outputs'
OUTPUT_DIR.mkdir(exist_ok=True)
ROOT = HERE
DATA = INPUT_DIR
OUTPUT = OUTPUT_DIR
__file__ = str(HERE / (FIGURE + '.ipynb'))
sys.argv = ['notebook']

def open_nc(path, mode='r', **kwargs):
    if 'memory' in kwargs:
        return _NETCDF_DATASET(path, mode, **kwargs)
    path = Path(path).resolve()
    previous = Path.cwd()
    try:
        os.chdir(path.parent)
        return _NETCDF_DATASET(path.name, mode, **kwargs)
    finally:
        os.chdir(previous)

def asset_path(key):
    if key in ['asset_e2603c619a67', 'asset_5e3c32038bfb']:
        return INPUT_DIR / 'study_mask.tif'
    if key == 'asset_beb53c9b3525':
        return INPUT_DIR / 'SPEI_k01-k24.nc'
    if key == 'asset_174bc324d489':
        return INPUT_DIR / 'SSMIrz_k01-k24.nc'
    return OUTPUT_DIR

def output_path(name=''):
    path = OUTPUT_DIR / name
    path.mkdir(parents=True, exist_ok=True)
    return path

def module_path(name=''):
    return str(HERE)

def font_path(filename):
    from matplotlib.font_manager import FontProperties, findfont
    family = 'Times New Roman' if filename.lower().startswith('times') else 'Courier New' if filename.lower().startswith('cour') else 'Arial'
    return findfont(FontProperties(family=family))

def array(value):
    return np.asarray(np.ma.filled(value.astype(float), np.nan))

@contextmanager
def mapped(path):
    with Path(path).open('rb') as stream:
        mm = mmap.mmap(stream.fileno(), 0, access=mmap.ACCESS_READ)
        dataset = open_nc('input.nc', memory=mm)
        try:
            yield dataset
        finally:
            dataset.close()
            gc.collect()
            mm.close()


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm, Normalize
from matplotlib.patches import Rectangle
import netCDF4 as nc
import rasterio
import warnings
SEASON_ORDER = ['MAM', 'JJA', 'SON', 'DJF']
ZONE_ORDER = ['Temperate', 'Tropical', 'Boreal', 'Arid']
RMAX_LABELS_ASC = ['-1.0--0.8', '-0.8--0.6', '-0.6--0.4', '-0.4--0.2', '-0.2-0.0', '0.0-0.2', '0.2-0.4', '0.4-0.6', '0.6-0.8', '0.8-1.0']
RMAX_LABELS_DISPLAY = ['0.8 - 1.0', '0.6 - 0.8', '0.4 - 0.6', '0.2 - 0.4', '0.0 - 0.2', '-0.2 - 0.0', '-0.4 - -0.2', '-0.6 - -0.4', '-0.8 - -0.6', '-1.0 - -0.8']
RMAX_LABELS_ORDER = list(reversed(RMAX_LABELS_ASC))
TS_LABELS = ['1', '2-3', '3-6', '6-12', '12-18', '18-24']

def insert_blank_columns(labels: list[str], heat: np.ndarray, level: np.ndarray, pvals: np.ndarray):
    out_labels = []
    out_heat_cols = []
    out_level = []
    out_p_cols = []
    x_groups = []
    src_cursor = 0
    for zi, zone in enumerate(ZONE_ORDER):
        start = len(out_labels)
        for lab in labels:
            out_labels.append(lab)
            out_heat_cols.append(heat[:, src_cursor])
            out_level.append(level[src_cursor, :])
            out_p_cols.append(pvals[:, src_cursor])
            src_cursor += 1
        end = len(out_labels) - 1
        x_groups.append((zone, start, end))
        if zi != len(ZONE_ORDER) - 1:
            out_labels.append('')
            out_heat_cols.append(np.full(heat.shape[0], np.nan))
            out_level.append(np.full(level.shape[1], np.nan))
            out_p_cols.append(np.full(pvals.shape[0], np.nan))
    return (out_labels, np.column_stack(out_heat_cols), np.vstack(out_level), np.column_stack(out_p_cols), x_groups)

def style_no_frame(ax):
    for sp in ax.spines.values():
        sp.set_visible(False)
    ax.tick_params(axis='both', which='both', length=0)

def draw_cell_grid(ax, ncols, nrows, labels, cell_gap=0.5, lw=4):
    for i in range(ncols):
        if labels[i] == '':
            continue
        for j in range(nrows):
            ax.add_patch(Rectangle((i - cell_gap, j - cell_gap), 1 + 2 * cell_gap, 1 + 2 * cell_gap, linewidth=lw, edgecolor='white', facecolor='none', zorder=2))

def draw_sig_stars(ax, labels, p_values, star_strong=13, star_weak=10):
    nrows, ncols = p_values.shape
    for i in range(ncols):
        if labels[i] == '':
            continue
        for j in range(nrows):
            p = p_values[j, i]
            if not np.isfinite(p):
                continue
            if p < 0.05:
                ax.text(i, j, '*', ha='center', va='center', fontsize=star_strong, fontweight='bold', color='black', zorder=6)
            elif p < 0.1:
                ax.text(i, j, '*', ha='center', va='center', fontsize=star_weak, fontweight='bold', color='gray', zorder=6)

def draw_group_bars(ax, groups, rect_y, rect_height):
    for zone, start, end in groups:
        rect = Rectangle((start - 0.5, rect_y), end - start + 1, rect_height, edgecolor='white', facecolor='Gainsboro', linewidth=2, zorder=3, clip_on=False)
        ax.add_patch(rect)
        ax.text((start + end) / 2, rect_y + rect_height / 2, zone, ha='center', va='center', fontsize=12.5, color='black', fontweight='bold', zorder=4, clip_on=False)

def make_plot_ready(trend: pd.DataFrame, pathway: str, metric: str):
    if metric == 'Rmax':
        grades = RMAX_LABELS_ORDER
        display = dict(zip(RMAX_LABELS_ORDER, RMAX_LABELS_DISPLAY))
    else:
        grades = TS_LABELS
        display = {g: g for g in grades}
    x_keys = [(zone, grade) for zone in ZONE_ORDER for grade in grades]
    ncols = len(x_keys)
    nrows = len(SEASON_ORDER)
    heat = np.full((nrows, ncols), np.nan)
    pvals = np.full((nrows, ncols), np.nan)
    level = np.full((ncols, nrows), np.nan)
    for xi, (zone, grade) in enumerate(x_keys):
        for yi, season in enumerate(SEASON_ORDER):
            m = (trend['pathway'] == pathway) & (trend['metric'] == metric) & (trend['climate_zone'] == zone) & (trend['grade'] == grade) & (trend['season'] == season)
            if not m.any():
                continue
            row = trend.loc[m].iloc[0]
            heat[yi, xi] = row['linear_slope_percentage_points_per_year']
            pvals[yi, xi] = row['p_value']
            level[xi, yi] = row['mean_percent']
    grade_labels = [display[g] for g in grades]
    labels, heat, level, pvals, groups = insert_blank_columns(grade_labels, heat, level, pvals)
    return (labels, heat, level, pvals, groups)

def plot_one(trend: pd.DataFrame, pathway: str, metric: str, out_path: Path):
    mpl.rcParams['font.family'] = 'Arial'
    mpl.rcParams['pdf.fonttype'] = 42
    mpl.rcParams['ps.fonttype'] = 42
    labels, heat, level, pvals, groups = make_plot_ready(trend, pathway, metric)
    nrows = len(SEASON_ORDER)
    ncols = len(labels)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', category=RuntimeWarning)
        mean_all = np.nanmean(level, axis=1)
        err_all = np.nanstd(level, axis=1) / 100.0
    mean_vmin, mean_vmax = (0.0, 60.0)
    s_min, s_max = (20, 200)
    err_vmin, err_vmax = (0.0, 0.12)
    norm_E = Normalize(vmin=err_vmin, vmax=err_vmax)
    cmap_E = mpl.cm.inferno_r
    vmax_R = 0.9
    norm_R = TwoSlopeNorm(vmin=-vmax_R, vcenter=0.0, vmax=vmax_R)
    cmap_R = mpl.cm.PuOr_r
    cell_gap = 0.5
    extent = [-cell_gap, ncols - 1 + cell_gap, nrows - 1 + cell_gap, -cell_gap]
    fig_width = 13.6 if metric == 'Rmax' else 9.4
    fig, ax = plt.subplots(figsize=(fig_width, 3.25))
    imH = ax.imshow(heat, cmap=cmap_R, norm=norm_R, aspect='equal', alpha=0.85, extent=extent)
    bubble_y = -1.15
    for i in range(ncols):
        if labels[i] == '':
            continue
        mean_val = mean_all[i]
        err_val = err_all[i]
        if not (np.isfinite(mean_val) and np.isfinite(err_val)):
            continue
        m01 = (np.clip(mean_val, mean_vmin, mean_vmax) - mean_vmin) / (mean_vmax - mean_vmin + 1e-12)
        size = s_min + (s_max - s_min) * m01
        color = cmap_E(norm_E(np.clip(err_val, err_vmin, err_vmax)))
        ax.scatter(i, bubble_y, s=size, c=[color], edgecolors='black', linewidths=0.8, zorder=10, clip_on=False)
    ax.set_xticks(range(ncols))
    ax.set_xticklabels(labels, rotation=90, ha='center', fontsize=8.5)
    ax.set_yticks(range(nrows))
    ax.set_yticklabels(SEASON_ORDER, fontsize=10.5)
    ax.tick_params(axis='x', pad=2)
    ax.tick_params(axis='y', pad=6)
    ax.set_xlim(-cell_gap, ncols - 1 + cell_gap)
    ax.set_ylim(-1.75, nrows + 1.45)
    style_no_frame(ax)
    draw_cell_grid(ax, ncols=ncols, nrows=nrows, labels=labels, cell_gap=cell_gap, lw=4)
    draw_sig_stars(ax, labels, pvals, star_strong=16 if metric == 'Rmax' else 11, star_weak=13 if metric == 'Rmax' else 9)
    draw_group_bars(ax, groups, rect_y=nrows - 1 + cell_gap + 0.25, rect_height=0.38)
    ax_pos = ax.get_position()
    top_y = ax_pos.y1 + 0.055
    cbar_height = 0.012
    element_gap = 0.05 if metric == 'Ts' else 0.035
    cbar1_left = ax_pos.x0
    cbar1_width = 0.13 if metric == 'Rmax' else 0.12
    cax_bubble = fig.add_axes([cbar1_left, top_y, cbar1_width, cbar_height])
    smE = mpl.cm.ScalarMappable(norm=norm_E, cmap=cmap_E)
    cbE = fig.colorbar(smE, cax=cax_bubble, orientation='horizontal', ticks=[err_vmin, err_vmax])
    cbE.set_label('Seasonal STD', fontsize=10.5, labelpad=5)
    cbE.ax.xaxis.set_label_position('top')
    cbE.ax.xaxis.set_ticks_position('top')
    cbE.ax.tick_params(axis='x', length=0, pad=1)
    cbE.ax.set_xticklabels([f'{err_vmin:.2f}', f'{err_vmax:.2f}'], fontsize=8.5)
    cbar2_left = cbar1_left + cbar1_width + element_gap
    cbar2_width = 0.13 if metric == 'Rmax' else 0.12
    cax_heat = fig.add_axes([cbar2_left, top_y, cbar2_width, cbar_height])
    cbH = fig.colorbar(imH, cax=cax_heat, orientation='horizontal', ticks=[-vmax_R, 0, vmax_R], format='%0.1f')
    cbH.set_label('Area Change Rate', fontsize=10.5, labelpad=5)
    cbH.ax.xaxis.set_label_position('top')
    cbH.ax.xaxis.set_ticks_position('top')
    cbH.ax.tick_params(axis='x', length=0, pad=1)
    cbH.ax.set_xticklabels([f'{-vmax_R:.1f}', '0', f'{vmax_R:.1f}'], fontsize=8.5)
    legend_left = cbar2_left + cbar2_width + element_gap
    legend_width = 0.34
    ax_legend = fig.add_axes([legend_left, top_y - 0.005, legend_width, 0.04])
    ax_legend.axis('off')
    legend_vals = np.array([10.0, 30.0, 50.0])
    legend_sizes = s_min + (s_max - s_min) * ((legend_vals - mean_vmin) / (mean_vmax - mean_vmin + 1e-12))
    handles, legend_labels = ([], [])
    for vv, ss in zip(legend_vals, legend_sizes):
        h = ax_legend.scatter([], [], s=ss, c='lightyellow', edgecolors='black', linewidths=0.8)
        handles.append(h)
        legend_labels.append(f'{vv:.0f}%')
    ax_legend.legend(handles, legend_labels, title='Mean Area Proportion', frameon=False, fontsize=8.8, title_fontsize=10.5, loc='center left', ncol=3, handletextpad=0.4, columnspacing=0.9, borderaxespad=0)
    plt.subplots_adjust(left=0.075, right=0.965, top=0.78, bottom=0.25)
    fig.savefig(out_path, dpi=600, bbox_inches='tight', pad_inches=0.08)
    fig.savefig(out_path.with_suffix('.pdf'), bbox_inches='tight', pad_inches=0.08)
    plt.close(fig)
SEASON_ORDER = ['MAM']
ZONE_ORDER = ['Temperate']
TS_LABELS = ['1', '2-3', '4-6', '7-12', '13-18', '19-24']
trend = pd.read_csv(INPUT_DIR / 'class_trends.csv')
for pathway in ['SPEI', 'SSMIrz']:
    plot_one(trend, pathway, 'Rmax', OUTPUT_DIR / f'Ropt_{pathway}_600dpi.png')


In [ ]:
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Rectangle
from matplotlib.colors import Normalize
from scipy.stats import linregress
from scipy.interpolate import PchipInterpolator
import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.colors import Normalize
from matplotlib.patches import Rectangle, Patch
import matplotlib.patheffects as pe
from matplotlib.lines import Line2D
from matplotlib.legend_handler import HandlerBase
from matplotlib.text import Text
from scipy.interpolate import PchipInterpolator

def load_matrix(data_path, file_names, row_slices=((0, 21), (35, 42))):
    mean_values = []
    p_values = []
    climate_zones = None
    for file in file_names:
        df = pd.read_csv(f'{data_path}{file}')
        if climate_zones is None:
            cz = [df.iloc[a:b, 1].values for a, b in row_slices]
            climate_zones = np.concatenate(cz)
        vals = [df.iloc[a:b, 2].values for a, b in row_slices]
        ps = [df.iloc[a:b, 3].values for a, b in row_slices]
        mean_values.append(np.concatenate(vals))
        p_values.append(np.concatenate(ps))
    mean_values = np.array(mean_values).T
    p_values = np.array(p_values).T
    return (climate_zones.tolist(), mean_values, p_values)

def process_zone_labels(climate_zones):
    out = []
    for zone in climate_zones:
        parts = str(zone).split('[')
        if len(parts) > 1:
            bracket_content = parts[1].split(')')[0]
            out.append(bracket_content.replace(',', ' -'))
        else:
            out.append(str(zone))
    return out

def insert_blank_rows(climate_zones, A, B, insert_positions=(7, 14, 21)):
    climate_zones = climate_zones.copy()
    A = A.copy()
    B = B.copy()
    for pos in sorted(insert_positions, reverse=True):
        climate_zones.insert(pos, '')
        A = np.insert(A, pos, np.nan, axis=0)
        B = np.insert(B, pos, np.nan, axis=0)
    return (climate_zones, A, B)

def style_no_frame(ax):
    for sp in ax.spines.values():
        sp.set_visible(False)
    ax.tick_params(axis='both', which='both', length=0)

def auto_absmax_scale(M, zones_cols=None, eps=1e-12):
    X = np.array(M, dtype=float, copy=True)
    if zones_cols is not None:
        blank_mask = np.array([z == '' for z in zones_cols])
        if blank_mask.any():
            X[:, blank_mask] = np.nan
    m = np.nanmax(np.abs(X))
    if not np.isfinite(m) or m <= 0:
        m = 1.0
    return float(m + eps)

def find_continuous_segments(mask):
    segments = []
    in_segment = False
    start = 0
    for i, valid in enumerate(mask):
        if valid and (not in_segment):
            start = i
            in_segment = True
        elif not valid and in_segment:
            segments.append((start, i))
            in_segment = False
    if in_segment:
        segments.append((start, len(mask)))
    return segments

def draw_ridgeline(ax, M_rows_cols, zones_cols, scale=None, height=0.55, line_color='#222222', pos_color='#D95F02', neg_color='#483D8B', alpha_fill=0.55, lw=1.15, zorder=2, label_prefix='', use_pchip=True, pchip_points=300, linestyle='-'):
    nrows, ncols = M_rows_cols.shape
    x = np.arange(ncols)
    y_pos = np.full_like(M_rows_cols, np.nan, dtype=float)
    blank_mask = np.array([z == '' for z in zones_cols])
    if scale is None:
        scale = auto_absmax_scale(M_rows_cols, zones_cols=zones_cols)
    for j in range(nrows):
        v = M_rows_cols[j, :].astype(float).copy()
        v[blank_mask] = np.nan
        baseline = float(j)
        ax.plot([x[0] - 0.5, x[-1] + 0.5], [baseline, baseline], color='#ECECEC', lw=1.0, zorder=zorder - 1)
        valid_mask = np.isfinite(v)
        segments = find_continuous_segments(valid_mask)
        first_segment = True
        for seg_start, seg_end in segments:
            if seg_end - seg_start < 2:
                continue
            x_seg = x[seg_start:seg_end]
            v_seg = v[seg_start:seg_end]
            y_seg = baseline + height * (v_seg / scale)
            y_pos[j, seg_start:seg_end] = y_seg + 0.08
            label_line = f'{label_prefix}' if j == 0 and label_prefix and first_segment else None
            first_segment = False
            if use_pchip and len(x_seg) >= 3:
                interp = PchipInterpolator(x_seg, y_seg)
                x_smooth = np.linspace(x_seg[0], x_seg[-1], int(pchip_points))
                y_smooth = interp(x_smooth)
                ax.plot(x_smooth, y_smooth, color=line_color, lw=lw, linestyle=linestyle, zorder=zorder + 1, label=label_line)
                y0 = np.full_like(y_smooth, baseline)
                y_pos_fill = np.where(y_smooth >= baseline, y_smooth, np.nan)
                y_neg_fill = np.where(y_smooth < baseline, y_smooth, np.nan)
                ax.fill_between(x_smooth, y0, y_pos_fill, where=np.isfinite(y_pos_fill), color=pos_color, alpha=alpha_fill, zorder=zorder)
                ax.fill_between(x_smooth, y0, y_neg_fill, where=np.isfinite(y_neg_fill), color=neg_color, alpha=alpha_fill, zorder=zorder)
            else:
                ax.plot(x_seg, y_seg, color=line_color, lw=lw, linestyle=linestyle, zorder=zorder + 1, label=label_line)
                y0 = np.full_like(y_seg, baseline)
                y_pos_fill = np.where(v_seg >= 0, y_seg, np.nan)
                y_neg_fill = np.where(v_seg < 0, y_seg, np.nan)
                ax.fill_between(x_seg, y0, y_pos_fill, where=np.isfinite(y_pos_fill), color=pos_color, alpha=alpha_fill, zorder=zorder)
                ax.fill_between(x_seg, y0, y_neg_fill, where=np.isfinite(y_neg_fill), color=neg_color, alpha=alpha_fill, zorder=zorder)
    return (y_pos, scale)

def draw_sig_stars_dual(ax, zones_cols, p_rows_cols_1, p_rows_cols_2, y_pos_rows_cols_1=None, y_pos_rows_cols_2=None, star_strong_1=19, star_weak_1=15, star_color_strong_1='black', star_color_weak_1='0.35', star_symbol_1='*', star_strong_2=14, star_weak_2=11, star_color_strong_2='#2C7BB6', star_color_weak_2='#7FCDBB', star_symbol_2='●', offset=0.15, zorder_1=6, zorder_2=7):
    nrows, ncols = p_rows_cols_1.shape
    for i in range(ncols):
        if zones_cols[i] == '':
            continue
        for j in range(nrows):
            p1 = p_rows_cols_1[j, i]
            p2 = p_rows_cols_2[j, i]
            sig1 = np.isfinite(p1) and p1 < 0.1
            sig2 = np.isfinite(p2) and p2 < 0.1
            baseline = float(j)
            if sig1 and (not sig2):
                yy = baseline if y_pos_rows_cols_1 is None else y_pos_rows_cols_1[j, i]
                if not np.isfinite(yy):
                    yy = baseline
                fs = star_strong_1 if p1 < 0.05 else star_weak_1
                col = star_color_strong_1 if p1 < 0.05 else star_color_weak_1
                t = ax.text(i, yy, star_symbol_1, ha='center', va='center', fontsize=fs, fontweight='bold', color=col, zorder=zorder_1)
                t.set_path_effects([pe.withStroke(linewidth=2.5, foreground='white')])
            elif sig2 and (not sig1):
                yy = baseline if y_pos_rows_cols_2 is None else y_pos_rows_cols_2[j, i]
                if not np.isfinite(yy):
                    yy = baseline
                fs = star_strong_2 if p2 < 0.05 else star_weak_2
                col = star_color_strong_2 if p2 < 0.05 else star_color_weak_2
                t = ax.text(i, yy, star_symbol_2, ha='center', va='center', fontsize=fs, fontweight='bold', color=col, zorder=zorder_2)
                t.set_path_effects([pe.withStroke(linewidth=2.5, foreground='white')])
            elif sig1 and sig2:
                yy1 = baseline if y_pos_rows_cols_1 is None else y_pos_rows_cols_1[j, i]
                if not np.isfinite(yy1):
                    yy1 = baseline
                yy1 = max(yy1, baseline) + offset * 0.3
                fs1 = star_strong_1 if p1 < 0.05 else star_weak_1
                col1 = star_color_strong_1 if p1 < 0.05 else star_color_weak_1
                t1 = ax.text(i, yy1, star_symbol_1, ha='center', va='center', fontsize=fs1, fontweight='bold', color=col1, zorder=zorder_1)
                t1.set_path_effects([pe.withStroke(linewidth=2.5, foreground='white')])
                yy2 = baseline if y_pos_rows_cols_2 is None else y_pos_rows_cols_2[j, i]
                if not np.isfinite(yy2):
                    yy2 = baseline
                yy2 = min(yy2, baseline) - offset * 0.3
                fs2 = star_strong_2 if p2 < 0.05 else star_weak_2
                col2 = star_color_strong_2 if p2 < 0.05 else star_color_weak_2
                t2 = ax.text(i, yy2, star_symbol_2, ha='center', va='center', fontsize=fs2, fontweight='bold', color=col2, zorder=zorder_2)
                t2.set_path_effects([pe.withStroke(linewidth=2.5, foreground='white')])

def draw_group_bar_horizontal_auto(ax, zones_cols, rect_y, rect_height, group_labels=('Temperate', 'Tropical', 'Boreal', 'Arid'), fontsize=12):
    n = len(zones_cols)
    blanks = [i for i, z in enumerate(zones_cols) if z == '']
    starts = [0] + [b + 1 for b in blanks]
    ends = blanks + [n]
    rect_params = []
    for s, e in zip(starts, ends):
        if e <= s:
            continue
        x0 = s - 0.5
        w = e - s
        rect_params.append((x0, rect_y, w, rect_height))
    for x, y, w, h in rect_params:
        ax.add_patch(Rectangle((x, y), w, h, edgecolor='white', facecolor='0.94', linewidth=2, linestyle='-', zorder=3, clip_on=False))
    k = min(len(rect_params), len(group_labels))
    for i in range(k):
        x, y, w, h = rect_params[i]
        ax.text(x + w / 2, y + h / 2, group_labels[i], rotation=0, ha='center', va='center', fontsize=fontsize, color='black', fontweight='bold', zorder=4, clip_on=False)

class TextStarHandler(HandlerBase):

    def __init__(self, color='black', size_multiplier=1.0):
        self.color = color
        self.size_multiplier = size_multiplier
        super().__init__()

    def create_artists(self, legend, orig_handle, xdescent, ydescent, width, height, fontsize, trans):
        x = width / 2
        y = height / 2
        actual_fontsize = fontsize * self.size_multiplier
        text = Text(x, y, '*', fontsize=actual_fontsize, ha='center', va='center', fontweight='bold', color=self.color, transform=trans)
        return [text]

class TextCircleHandler(HandlerBase):

    def __init__(self, color='#2C7BB6', size_multiplier=1.0):
        self.color = color
        self.size_multiplier = size_multiplier
        super().__init__()

    def create_artists(self, legend, orig_handle, xdescent, ydescent, width, height, fontsize, trans):
        x = width / 2
        y = height / 2
        actual_fontsize = fontsize * self.size_multiplier
        text = Text(x, y, '●', fontsize=actual_fontsize, ha='center', va='center', fontweight='bold', color=self.color, transform=trans)
        return [text]

def draw_ridgeline(ax, M_rows_cols, zones_cols, scale=None, height=0.75, line_color='#222222', pos_color='#D95F02', neg_color='#483D8B', alpha_fill=0.55, lw=1.15, zorder=2, label_prefix='', use_pchip=True, pchip_points=350, linestyle='-'):
    values = np.asarray(M_rows_cols, float)
    if scale is None:
        scale = auto_absmax_scale(values, zones_cols)
    out = np.full_like(values, np.nan)
    x = np.arange(values.shape[1])
    for j in range(values.shape[0]):
        ax.plot([-0.5, len(x) - 0.5], [j, j], color='#ECECEC', lw=1, zorder=zorder - 1)
        valid = np.isfinite(values[j]) & np.array([z != '' for z in zones_cols])
        for a, b in find_continuous_segments(valid):
            xx = x[a:b]
            v = values[j, a:b]
            yy = j + height * v / scale
            out[j, a:b] = yy
            if use_pchip and len(xx) >= 3:
                f = PchipInterpolator(xx, yy)
                xs = np.unique(np.concatenate([np.linspace(xx[0], xx[-1], pchip_points), xx]))
                ys = f(xs)
                assert np.allclose(f(xx), yy, atol=1e-14, rtol=0)
            else:
                xs, ys = (xx, yy)
            ax.plot(xs, ys, color=line_color, lw=lw, linestyle=linestyle, zorder=zorder + 1)
            ax.fill_between(xs, j, ys, where=ys >= j, interpolate=True, color=pos_color, alpha=alpha_fill, zorder=zorder)
            ax.fill_between(xs, j, ys, where=ys <= j, interpolate=True, color=neg_color, alpha=alpha_fill, zorder=zorder)
    return (out, scale)

def draw_sig_stars_dual(ax, zones_cols, p_rows_cols_1, p_rows_cols_2, y_pos_rows_cols_1=None, y_pos_rows_cols_2=None, **kwargs):
    for j in range(p_rows_cols_1.shape[0]):
        for i, z in enumerate(zones_cols):
            if not z:
                continue
            p1 = p_rows_cols_1[j, i]
            p2 = p_rows_cols_2[j, i]
            y1 = y_pos_rows_cols_1[j, i]
            y2 = y_pos_rows_cols_2[j, i]
            if np.isfinite(p1) and p1 < 0.1:
                artist = ax.scatter(i, y1, marker='$\\ast$', s=100 if p1 < 0.05 else 65, color='black' if p1 < 0.05 else '0.35', edgecolors='white', linewidths=0.35, zorder=9)
                assert np.allclose(artist.get_offsets()[0], [i, y1], atol=0, rtol=0)
            if np.isfinite(p2) and p2 < 0.1:
                artist = ax.scatter(i, y2, s=32 if p2 < 0.05 else 22, color='#2C7BB6' if p2 < 0.05 else '#7FCDBB', edgecolors='white', linewidths=0.8, zorder=10)
                assert np.allclose(artist.get_offsets()[0], [i, y2], atol=0, rtol=0)
OUT = OUTPUT_DIR
SEASONS = ['MAM']
ZONES = ['Temperate']
PATHS = ['SPEI', 'SSMIrz']
BINS = [('19-24', 19, 24), ('13-18', 13, 18), ('7-12', 7, 12), ('4-6', 4, 6), ('2-3', 2, 3), ('1', 1, 1)]

def main():
    OUT.mkdir(parents=True, exist_ok=True)
    source = INPUT_DIR / 'class_trends.csv'
    rawsource = INPUT_DIR / 'area_shares.csv'
    table = pd.read_csv(source)
    raw = pd.read_csv(rawsource)
    table = table[table.metric == 'Ts'].copy()
    raw = raw[raw.metric == 'Ts'].copy()
    keys = ['pathway', 'climate_zone', 'grade', 'season']
    assert not table.duplicated(keys).any() and len(table) == 12
    assert set(table.grade) == {b[0] for b in BINS}
    errors = []
    indexed = table.set_index(keys)
    for key, group in raw.groupby(keys):
        g = group.sort_values('window_index')
        assert len(g) == 11
        assert g.window.tolist() == [f'{y}-{y + 9}' for y in range(2001, 2012)]
        fit = linregress(g.window_index.to_numpy(float), g.percent.to_numpy(float))
        r = indexed.loc[key]
        assert abs(fit.slope - r.linear_slope_percentage_points_per_year) < 1e-11
        assert abs(fit.pvalue - r.p_value) < 1e-10
        assert abs(g.percent.mean() - r.mean_percent) < 1e-10
        errors.append(abs(fit.slope - r.linear_slope_percentage_points_per_year))
    sums = raw.groupby(['pathway', 'climate_zone', 'window', 'season']).percent.sum()
    assert len(sums) == 22 and np.allclose(sums, 100, atol=1e-09)
    cols = []
    for zi, zone in enumerate(ZONES):
        if zi:
            cols.append(None)
        cols.extend(((zone, b[0]) for b in BINS))
    labels = ['' if col is None else col[1] for col in cols]
    values = {}
    pvals = {}
    levels = {}
    rows = []
    for path in PATHS:
        values[path] = np.full((len(SEASONS), len(cols)), np.nan)
        pvals[path] = values[path].copy()
        levels[path] = values[path].copy()
        for ci, col in enumerate(cols):
            if col is None:
                continue
            zone, grade = col
            for si, season in enumerate(SEASONS):
                r = indexed.loc[path, zone, grade, season]
                values[path][si, ci] = r.linear_slope_percentage_points_per_year
                pvals[path][si, ci] = r.p_value
                levels[path][si, ci] = r.mean_percent / 100
                rows.append({'column': ci, 'pathway': path, 'climate_zone': zone, 'grade': grade, 'season': season, 'slope_percentage_points_per_year': values[path][si, ci], 'p_value': pvals[path][si, ci], 'mean_area_proportion': levels[path][si, ci]})
    members = [j for _, lo, hi in BINS for j in range(lo, hi + 1)]
    assert sorted(members) == list(range(1, 25)) and len(set(members)) == 24
    vmax = max((np.nanmax(abs(a)) for a in values.values()))
    mpl.rcParams.update({'font.family': 'Arial', 'font.size': 12, 'axes.titlesize': 12, 'axes.labelsize': 12, 'xtick.labelsize': 12, 'ytick.labelsize': 12, 'legend.fontsize': 12, 'pdf.fonttype': 42, 'ps.fonttype': 42, 'savefig.dpi': 600, 'figure.facecolor': 'white', 'axes.facecolor': 'white'})
    fig = plt.figure(figsize=(14, 3.4))
    gs = fig.add_gridspec(4, 1, height_ratios=[0.4, 0.5, 5.5, 0.6], hspace=0.3)
    ag = fig.add_subplot(gs[0])
    at = fig.add_subplot(gs[1], sharex=ag)
    ar = fig.add_subplot(gs[2], sharex=ag)
    ab = fig.add_subplot(gs[3], sharex=ag)
    for ax in [ag, at, ar, ab]:
        style_no_frame(ax)
        ax.set_xlim(-0.5, len(cols) - 0.5)
    ag.set_ylim(0, 1)
    ag.set_yticks([])
    ag.tick_params(axis='x', labelbottom=False)
    draw_group_bar_horizontal_auto(ag, labels, 0.2, 0.6, tuple(ZONES), fontsize=16)
    norm = Normalize(0, 0.07)
    cmap = mpl.colormaps['inferno_r']
    bubble_rows = []
    for ax, path in [(at, 'SSMIrz'), (ab, 'SPEI')]:
        ax.set_ylim(-0.6, 0.6)
        ax.set_yticks([])
        ax.text(-0.6, 0, 'SSMI$_{rz}$' if path == 'SSMIrz' else 'SPEI', ha='right', va='center', fontsize=13)
        for ci, col in enumerate(cols):
            if col is None:
                continue
            mean = levels[path][:, ci].mean()
            std = levels[path][:, ci].std(ddof=0)
            size = 20 + 180 * np.clip(mean / 0.6, 0, 1)
            ax.scatter(ci, 0, s=size, c=[cmap(norm(std))], edgecolors='black', linewidths=0.8, zorder=5)
            bubble_rows.append({'pathway': path, 'column': ci, 'climate_zone': col[0], 'grade': col[1], 'mean_area_proportion': mean, 'seasonal_std': std})
    at.tick_params(axis='x', labelbottom=False)
    pos1, _ = draw_ridgeline(ar, values['SPEI'], labels, scale=vmax, height=0.75, line_color='#964800', pos_color='#D95F02', neg_color='#483D8B', alpha_fill=0.55, lw=1.15, zorder=2, use_pchip=True, pchip_points=350, linestyle='-')
    pos2, _ = draw_ridgeline(ar, values['SSMIrz'], labels, scale=vmax, height=0.75, line_color='#420097', pos_color='#D95F02', neg_color='#483D8B', alpha_fill=0.45, lw=1.2, zorder=4, use_pchip=True, pchip_points=350, linestyle='--')
    draw_sig_stars_dual(ar, labels, pvals['SPEI'], pvals['SSMIrz'], pos1, pos2, offset=0.15)
    ar.set_yticks(range(len(SEASONS)))
    ar.set_yticklabels(SEASONS, fontsize=14)
    ar.tick_params(axis='x', labelbottom=False)
    ar.set_ylim(-0.85, len(SEASONS) - 0.1)
    ab.set_xticks(range(len(cols)))
    ab.set_xticklabels(labels, rotation=90, ha='center', fontsize=12)
    dummies = [Rectangle((0, 0), 1, 1, facecolor='none', edgecolor='none') for _ in range(4)]
    legend = [Line2D([], [], color='#964800', lw=1.15), Line2D([], [], color='#420097', lw=1.2, ls='--'), Patch(facecolor='#D95F02', alpha=0.55), Patch(facecolor='#483D8B', alpha=0.55)] + dummies
    handlers = {dummies[0]: TextStarHandler('black', 19 / 12), dummies[1]: TextStarHandler('0.35', 15 / 12), dummies[2]: TextCircleHandler('#2C7BB6', 14 / 12), dummies[3]: TextCircleHandler('#7FCDBB', 11 / 12)}
    fig.legend(legend, ['SPEI–SIF', 'SSMI$_{rz}$–SIF', 'Increasing area share', 'Decreasing area share', 'SPEI–SIF p<0.05', 'SPEI–SIF 0.05≤p<0.1', 'SSMI$_{rz}$–SIF p<0.05', 'SSMI$_{rz}$–SIF 0.05≤p<0.1'], handler_map=handlers, loc='lower center', bbox_to_anchor=(0.52, 1.02), ncol=4, frameon=False, columnspacing=1, handlelength=1.5, handletextpad=0.5)
    fig.subplots_adjust(left=0.08, right=0.98, top=0.96, bottom=0.16)
    cax = fig.add_axes([0.09, -0.22, 0.18, 0.022])
    cb = fig.colorbar(mpl.cm.ScalarMappable(norm=norm, cmap=cmap), cax=cax, orientation='horizontal', ticks=[0, 0.07])
    cb.ax.set_title('Seasonal STD', fontsize=11, pad=5)
    cb.ax.tick_params(labelsize=10)
    handles = [plt.scatter([], [], s=20 + 180 * v / 0.6, c='#ffffb2', edgecolors='black', linewidths=0.8) for v in [0.1, 0.3, 0.5]]
    fig.legend(handles, ['10%', '30%', '50%'], title='Mean area proportion', loc='upper center', bbox_to_anchor=(0.52, -0.16), ncol=3, frameon=False, fontsize=11, title_fontsize=11, columnspacing=0.7)
    fig.text(0.74, -0.2, f'Ridge amplitude: ±{vmax:.3f} pp/year\nTime-scale classes (months)', ha='left', va='top', fontsize=10)
    nonblank = np.array([c is not None for c in cols])
    for path, positions in [('SPEI', pos1), ('SSMIrz', pos2)]:
        expected = np.arange(len(SEASONS))[:, None] + 0.75 * values[path] / vmax
        assert np.allclose(positions[:, nonblank], expected[:, nonblank], atol=1e-14, rtol=0)
        assert np.nanmin(positions) > ar.get_ylim()[0] and np.nanmax(positions) < ar.get_ylim()[1]
    for path in PATHS:
        for zi in range(len(ZONES)):
            ix = np.arange(zi * 7, zi * 7 + 6)
            for si in range(len(SEASONS)):
                v = values[path][si, ix]
                f = PchipInterpolator(ix, v)
                assert np.allclose(f(ix), v, atol=1e-14, rtol=0)
    prefix = OUT / 'Ts_smoothed_curves_significance_at_original_nodes_four_climate_zones_600dpi'
    fig.savefig(str(prefix) + '.png', dpi=600, bbox_inches='tight', pad_inches=0.12)
    fig.savefig(str(prefix) + '.pdf', dpi=600, bbox_inches='tight', pad_inches=0.12)
    plt.close(fig)
    pd.DataFrame(rows).to_csv(OUT / 'ridgeline_significance_pointwise_plot_data.csv', index=False, encoding='utf-8-sig')
    pd.DataFrame(bubble_rows).to_csv(OUT / 'top_and_bottom_bubbles_pointwise_plot_data.csv', index=False, encoding='utf-8-sig')
main()
